In [ ]:
import sys
!{sys.executable} -m pip install transformers 

You should consider upgrading via the 'c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import GPT2Tokenizer
import math
import copy

# ==========================================
# 1. Diffusion Utilities (Sqrt Schedule)
# ==========================================
def compute_sqrt_alpha_bar(t, T_max=2000, s=1e-4):
    """
    Sqrt noise schedule from Diffusion-LM (Appendix A).
    alpha_bar_t = 1 - sqrt(t/T + s)
    """
    # t is expected to be a tensor of shape (batch_size, 1, 1)
    t_norm = t.float() / T_max
    alpha_bar = 1.0 - torch.sqrt(t_norm + s)
    # Clamp to prevent negative variances or exact zeros
    return torch.clamp(alpha_bar, min=1e-5, max=0.9999)

class EMA:
    """Exponential Moving Average for model weights (Crucial for TRM stability)"""
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]

In [15]:
# ==========================================
# 2. Model Architecture (TRM + Diffusion)
# ==========================================
class BidirectionalTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.norm1 = nn.LayerNorm(d_model)
        
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        
        x = self.norm1(x + self.out_proj(attn_out))
        x = self.norm2(x + self.mlp(x))
        return x

class TRM_Diffusion(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, d_ff=2048, num_layers=4):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        
        # End-to-End Embeddings
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(1024, d_model) 
        
        # Timestep embedding (Diffusion specific)
        self.time_mlp = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model)
        )
        
        # TRM Backbone (Less is More - keeping layers relatively low)
        self.layers = nn.ModuleList([
            BidirectionalTransformerLayer(d_model, n_heads, d_ff) 
            for _ in range(num_layers)
        ])
        
        # Continuous Output Head (Predicting x_0)
        self.output_head = nn.Linear(d_model, d_model)
        self.z_init = nn.Parameter(torch.randn(1, 1, d_model))

        # Halting Head (ACT)
        self.q_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, 1),
            nn.Sigmoid() 
        )

    def get_sinusoidal_embeddings(self, t):
        """Standard sinusoidal positional embeddings for time"""
        half_dim = self.d_model // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

    def latent_recursion(self, x_t_emb, z, t_emb, n):
        """Inner reasoning loop"""
        for _ in range(n):
            # Combine current noisy observation, latent thought, and time
            combined_state = x_t_emb + z + t_emb.unsqueeze(1)
            
            for layer in self.layers:
                combined_state = layer(combined_state)
            z = combined_state
            
        pred_x0_emb = self.output_head(z)
        return pred_x0_emb, z

    def deep_recursion(self, x_t_emb, z, t, n=6, T=3):
        """Deep Supervision loop mimicking pseudo-depth"""
        t_emb = self.time_mlp(self.get_sinusoidal_embeddings(t))
        
        # 1. Thinking Phase (No Gradients)
        with torch.no_grad():
            for _ in range(T - 1):
                _, z = self.latent_recursion(x_t_emb, z, t_emb, n)
                
        # 2. Action Phase (Track Gradients)
        pred_x0_emb, z = self.latent_recursion(x_t_emb, z, t_emb, n)
        
        z_mean = z.mean(dim=1) 
        q_hat = self.q_head(z_mean).squeeze(-1)
        
        return pred_x0_emb, z.detach(), q_hat

In [5]:
# ==========================================
# 3. Dataset & Dataloader 
# ==========================================
def get_rocstories_dataloader(batch_size=64, seq_len=64):
    """Loads ROCStories and tokenizes them"""
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    
    # We use a subset/variant of ROCStories available on HuggingFace
    dataset = load_dataset("roneneldan/TinyStories", split="train[:10000]") 
    
    def tokenize(batch):
        # TinyStories just uses the "text" column
        return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=seq_len, return_tensors="pt")
    
    dataset = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
    dataset.set_format(type="torch", columns=["input_ids"])
    return DataLoader(dataset, batch_size=batch_size, shuffle=True), len(tokenizer)

In [5]:
# ==========================================
# 4. Training Loop (N_SUP = 1, No ACT Halting Loss)
# ==========================================
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seq_len = 128
    batch_size = 16
    T_DIFFUSION = 2000
    
    # TRM Hyperparameters
    N_RECURSIONS = 6  # 'n' in TRM
    T_CYCLES = 3      # 'T' in TRM (2 thought loops, 1 action loop)
    
    dataloader, vocab_size = get_rocstories_dataloader(batch_size, seq_len)
    
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    ema = EMA(model, decay=0.999)
    
    print("Starting Training...")
    model.train()
    
    for epoch in range(10): 
        for step, batch in enumerate(dataloader):
            input_ids = batch["input_ids"].to(device)
            bs = input_ids.shape[0]
            
            # 1. Clean x_0 Embeddings
            y_true_emb = model.token_emb(input_ids)
            positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(bs, seq_len)
            y_true_emb = y_true_emb + model.pos_emb(positions)
            
            # 2. Sample random timestep t and apply Sqrt Noise
            t = torch.randint(1, T_DIFFUSION + 1, (bs,), device=device)
            alpha_bar_t = compute_sqrt_alpha_bar(t, T_max=T_DIFFUSION).view(bs, 1, 1)
            noise = torch.randn_like(y_true_emb)
            
            # Forward Process q(x_t | x_0)
            x_t_emb = torch.sqrt(alpha_bar_t) * y_true_emb + torch.sqrt(1 - alpha_bar_t) * noise
            
            # 3. Initialize Latent state
            z = model.z_init.expand(bs, seq_len, -1)
            
            # 4. TRM Deep Recursion (Executes ONCE per batch)
            # We ignore q_hat (_) since the ACT halting mechanism is removed
            pred_x0_emb, z, _ = model.deep_recursion(x_t_emb, z, t, n=N_RECURSIONS, T=T_CYCLES)
            
            # 5. Calculate Diffusion-LM Losses
            loss_recon = F.mse_loss(pred_x0_emb, y_true_emb)
            loss_emb_anchor = F.mse_loss(y_true_emb, pred_x0_emb.detach())
            
            # Total loss (No halting loss included)
            total_loss = loss_recon + (0.1 * loss_emb_anchor)
            
            # 6. Backpropagate
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ema.update()
            
            if step % 50 == 0:
                print(f"Epoch {epoch} | Step {step} | Loss: {total_loss.item():.4f}")

    # ==========================================
        # 7. SAVE THE MODEL AT THE END OF THE EPOCH
        # ==========================================
        print(f"--- Epoch {epoch} Complete ---")
        
        # A. Apply the stable EMA weights to the model before saving
        ema.apply_shadow()
        
        # B. Create a dictionary with everything you need to resume training or run inference
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }
        
        # C. Save the file
        save_path = f"trm_diffusion_epoch_{epoch}.pt"
        torch.save(checkpoint, save_path)
        print(f"Saved checkpoint to {save_path}")
        
        # D. Restore the raw training weights so the next epoch can continue learning
        ema.restore()

if __name__ == "__main__":
    train()

Starting Training...
Epoch 0 | Step 0 | Loss: 2.5666
Epoch 0 | Step 50 | Loss: 1.9215
Epoch 0 | Step 100 | Loss: 1.4430
Epoch 0 | Step 150 | Loss: 1.4284
Epoch 0 | Step 200 | Loss: 1.2286
Epoch 0 | Step 250 | Loss: 1.0411
Epoch 0 | Step 300 | Loss: 0.9338
Epoch 0 | Step 350 | Loss: 1.2064
Epoch 0 | Step 400 | Loss: 0.7384
Epoch 0 | Step 450 | Loss: 0.6878
Epoch 0 | Step 500 | Loss: 0.6967
Epoch 0 | Step 550 | Loss: 0.6495
Epoch 0 | Step 600 | Loss: 0.6014
--- Epoch 0 Complete ---
Saved checkpoint to trm_diffusion_epoch_0.pt
Epoch 1 | Step 0 | Loss: 0.5759
Epoch 1 | Step 50 | Loss: 0.8768
Epoch 1 | Step 100 | Loss: 0.5843
Epoch 1 | Step 150 | Loss: 0.4629
Epoch 1 | Step 200 | Loss: 0.6072
Epoch 1 | Step 250 | Loss: 0.5444
Epoch 1 | Step 300 | Loss: 0.5846
Epoch 1 | Step 350 | Loss: 0.6516
Epoch 1 | Step 400 | Loss: 0.5465
Epoch 1 | Step 450 | Loss: 0.6887
Epoch 1 | Step 500 | Loss: 0.4168
Epoch 1 | Step 550 | Loss: 0.6436
Epoch 1 | Step 600 | Loss: 0.5655
--- Epoch 1 Complete ---
Saved 

In [7]:
import torch
import torch.nn.functional as F
from transformers import GPT2Tokenizer
import math

# 1. Include your TRM_Diffusion class and BidirectionalTransformerLayer here
# (Copy them from your training script so the model can be instantiated)
# class BidirectionalTransformerLayer(nn.Module): ...
# class TRM_Diffusion(nn.Module): ...

def compute_sqrt_alpha_bar(t, T_max=2000, s=1e-4):
    """Sqrt noise schedule (matches the training script)"""
    t_norm = t.float() / T_max
    alpha_bar = 1.0 - torch.sqrt(t_norm + s)
    return torch.clamp(alpha_bar, min=1e-5, max=0.9999)

def clamp_to_nearest_word(pred_x0_emb, vocab_embeddings):
    """
    THE CLAMPING TRICK:
    Finds the closest actual word embedding for each predicted continuous vector.
    Using L2 distance: ||pred - vocab||^2
    """
    # pred_x0_emb shape: (batch_size, seq_len, d_model)
    # vocab_embeddings shape: (vocab_size, d_model)
    
    # Calculate Euclidean distance between predictions and all vocab words
    dists = torch.cdist(pred_x0_emb, vocab_embeddings, p=2) # shape: (batch, seq, vocab)
    
    # Find the index of the closest word for each position
    nearest_idx = dists.argmin(dim=-1) 
    
    # Snap the vector to exactly that word's embedding
    clamped_x0_emb = vocab_embeddings[nearest_idx]
    
    return clamped_x0_emb, nearest_idx

@torch.no_grad() # No gradients needed for generation!
def generate_text(model_path, seq_len=64, batch_size=1):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    T_DIFFUSION = 2000
    
    # 1. Load Tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    vocab_size = len(tokenizer)
    
    # 2. Load Model & Weights
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    vocab_embeddings = model.token_emb.weight # The learned word vectors
    
    print("Starting generation reverse loop...")
    
    # 3. Start with pure Gaussian Noise (x_T)
    x_t_emb = torch.randn(batch_size, seq_len, model.d_model, device=device)
    
    # 4. The Step-by-Step Denoising Loop (T down to 1)
    for t_step in reversed(range(1, T_DIFFUSION + 1)):
        # Tensor formatting for t
        t_tensor = torch.full((batch_size,), t_step, device=device, dtype=torch.long)
        
        # Initialize the fresh TRM reasoning latent 'z' for this diffusion step
        z = model.z_init.expand(batch_size, seq_len, -1)
        
        # A. TRM guesses the clean x_0 from the current noisy x_t
        pred_x0_emb, _, _ = model.deep_recursion(x_t_emb, z, t_tensor, n=6, T=3)
        
        # B. THE CLAMPING TRICK
        # Snap the predicted continuous vectors to the nearest actual word embeddings
        clamped_x0_emb, token_ids = clamp_to_nearest_word(pred_x0_emb, vocab_embeddings)
        
        # C. Sample the next step x_{t-1}
        if t_step > 1:
            # Get alpha_bar for the NEXT step down
            t_minus_1 = torch.full((batch_size,), t_step - 1, device=device, dtype=torch.long)
            alpha_bar_t_minus_1 = compute_sqrt_alpha_bar(t_minus_1, T_max=T_DIFFUSION).view(batch_size, 1, 1)
            
            # Re-noise the clamped prediction slightly to get x_{t-1}
            noise = torch.randn_like(clamped_x0_emb)
            x_t_emb = torch.sqrt(alpha_bar_t_minus_1) * clamped_x0_emb + torch.sqrt(1 - alpha_bar_t_minus_1) * noise
        else:
            # At t=0, we just keep the exact clamped x_0
            x_t_emb = clamped_x0_emb
            
        if t_step % 200 == 0:
            print(f"Denoising step {t_step}/{T_DIFFUSION} complete...")

    # 5. Decode the final token IDs back into readable English
    final_text = tokenizer.batch_decode(token_ids, skip_special_tokens=True)
    
    print("\n--- Generated Story ---")
    print(final_text[0])
    return final_text

if __name__ == "__main__":
    # Point this to whatever your latest saved checkpoint is named
    generate_text("trm_diffusion_epoch_9.pt")

Starting generation reverse loop...
Denoising step 2000/2000 complete...
Denoising step 1800/2000 complete...
Denoising step 1600/2000 complete...
Denoising step 1400/2000 complete...
Denoising step 1200/2000 complete...
Denoising step 1000/2000 complete...
Denoising step 800/2000 complete...
Denoising step 600/2000 complete...
Denoising step 400/2000 complete...
Denoising step 200/2000 complete...

--- Generated Story ---
 her saw She had her was mom, her She day The She was mom little was He to She in was her mom it and was's
 in the in little it mom She. the was and a in little mom her little a her the her little She day little her The mom went mom her little mom She saw


In [ ]:
import torch
import torch.nn.functional as F
from transformers import GPT2Tokenizer
import math

# 1. Make sure to import or define BidirectionalTransformerLayer 
# and TRM_Diffusion here exactly as they are in your train script.

def compute_sqrt_alpha_bar(t, T_max=2000, s=1e-4):
    t_norm = t.float() / T_max
    alpha_bar = 1.0 - torch.sqrt(t_norm + s)
    return torch.clamp(alpha_bar, min=1e-5, max=0.9999)

def clamp_to_nearest_word(pred_word_emb, vocab_embeddings):
    """Clamps continuous vectors to the exact discrete vocabulary vectors."""
    dists = torch.cdist(pred_word_emb, vocab_embeddings, p=2) 
    nearest_idx = dists.argmin(dim=-1) 
    clamped_word_emb = vocab_embeddings[nearest_idx]
    return clamped_word_emb, nearest_idx

@torch.no_grad()
def generate_text(model_path, seq_len=64, batch_size=1):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    T_DIFFUSION = 2000
    
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    vocab_size = len(tokenizer)
    
    # Load Model (Ensure you use TRM_Diffusion, not TRM_Diffusion_Core)
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    vocab_embeddings = model.token_emb.weight 
    
    # Pre-compute positional embeddings for the sequence
    positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(batch_size, seq_len)
    pos_embeddings = model.pos_emb(positions)
    
    print("Starting generation reverse loop...")
    
    # Start with pure Gaussian Noise (x_T)
    x_t_emb = torch.randn(batch_size, seq_len, model.d_model, device=device)
    
    for t_step in reversed(range(1, T_DIFFUSION + 1)):
        t_tensor = torch.full((batch_size,), t_step, device=device, dtype=torch.long)
        
        # A. Initialize latent thought 'z' for this step
        z = model.z_init.expand(batch_size, seq_len, -1)
        
        # B. Predict the combined clean embedding (word + pos) using deep_recursion
        pred_x0_combined, _, _ = model.deep_recursion(x_t_emb, z, t_tensor, n=6, T=3)
        
        # C. THE CLAMPING TRICK
        # 1. Subtract pos_emb to isolate the word embedding
        pred_word_emb = pred_x0_combined - pos_embeddings
        
        # 2. Snap to nearest actual word
        clamped_word_emb, token_ids = clamp_to_nearest_word(pred_word_emb, vocab_embeddings)
        
        # 3. Add pos_emb back to get the true x_0 for the diffusion math
        clamped_x0 = clamped_word_emb + pos_embeddings
        
        # D. TRUE DDPM POSTERIOR MATH (Appendix E)
        if t_step > 1:
            t_minus_1 = torch.full((batch_size,), t_step - 1, device=device, dtype=torch.long)
            
            # Fetch alphas
            alpha_bar_t = compute_sqrt_alpha_bar(t_tensor).view(-1, 1, 1)
            alpha_bar_t_minus_1 = compute_sqrt_alpha_bar(t_minus_1).view(-1, 1, 1)
            alpha_t = alpha_bar_t / alpha_bar_t_minus_1
            beta_t = 1.0 - alpha_t
            
            # Calculate the posterior mean: mu = c1 * x_0 + c2 * x_t
            c1 = (torch.sqrt(alpha_bar_t_minus_1) * beta_t) / (1.0 - alpha_bar_t)
            c2 = (torch.sqrt(alpha_t) * (1.0 - alpha_bar_t_minus_1)) / (1.0 - alpha_bar_t)
            posterior_mean = c1 * clamped_x0 + c2 * x_t_emb
            
            # Calculate posterior variance
            posterior_variance = ((1.0 - alpha_bar_t_minus_1) / (1.0 - alpha_bar_t)) * beta_t
            
            # Add noise for the Langevin step
            noise = torch.randn_like(x_t_emb)
            x_t_emb = posterior_mean + torch.sqrt(posterior_variance) * noise
        else:
            # At t=1, we just lock in the final clamped x_0
            x_t_emb = clamped_x0
            
        if t_step % 200 == 0:
            print(f"Denoising step {t_step}/{T_DIFFUSION} complete...")

    # Decode the final token IDs
    final_text = tokenizer.batch_decode(token_ids, skip_special_tokens=True)
    
    print("\n--- Generated Story ---")
    print(final_text[0])
    return final_text

if __name__ == "__main__":
    generate_text("trm_diffusion_epoch_9.pt")

Starting generation reverse loop...
Denoising step 2000/2000 complete...
Denoising step 1800/2000 complete...
Denoising step 1600/2000 complete...
Denoising step 1400/2000 complete...
Denoising step 1200/2000 complete...
Denoising step 1000/2000 complete...
Denoising step 800/2000 complete...
Denoising step 600/2000 complete...
Denoising step 400/2000 complete...
Denoising step 200/2000 complete...

--- Generated Story ---
Once upon a time?". to wanted were home heard a who have his. had a was oldmy was,. a and the
, of to inside scared found found he the heard.my,.,. foundmy scared a to a a it itI him found outside, but so so they Lily the


In [8]:
import torch
import torch.nn.functional as F
from transformers import GPT2Tokenizer
import math

# 1. Ensure BidirectionalTransformerLayer and TRM_Diffusion are defined here!

def compute_sqrt_alpha_bar(t, T_max=2000, s=1e-4):
    t_norm = t.float() / T_max
    alpha_bar = 1.0 - torch.sqrt(t_norm + s)
    return torch.clamp(alpha_bar, min=1e-5, max=0.9999)

def clamp_to_nearest_word(pred_word_emb, vocab_embeddings):
    dists = torch.cdist(pred_word_emb, vocab_embeddings, p=2) 
    nearest_idx = dists.argmin(dim=-1) 
    clamped_word_emb = vocab_embeddings[nearest_idx]
    return clamped_word_emb, nearest_idx

@torch.no_grad()
def generate_text(model_path, seq_len=64, batch_size=1):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    T_DIFFUSION = 2000
    
    # NEW: Only clamp in the final 25% of denoising!
    CLAMP_START = 500 
    
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    vocab_size = len(tokenizer)
    
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    vocab_embeddings = model.token_emb.weight 
    positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(batch_size, seq_len)
    pos_embeddings = model.pos_emb(positions)
    
    print("Starting generation reverse loop...")
    
    # Start with pure Gaussian Noise (x_T)
    x_t_emb = torch.randn(batch_size, seq_len, model.d_model, device=device)
    token_ids = None # Will be populated once clamping starts
    
    for t_step in reversed(range(1, T_DIFFUSION + 1)):
        t_tensor = torch.full((batch_size,), t_step, device=device, dtype=torch.long)
        
        z = model.z_init.expand(batch_size, seq_len, -1)
        pred_x0_combined, _, _ = model.deep_recursion(x_t_emb, z, t_tensor, n=6, T=3)
        
        # --- DELAYED CLAMPING LOGIC ---
        if t_step <= CLAMP_START:
            # We are close to t=0, time to force the vectors into real words
            pred_word_emb = pred_x0_combined - pos_embeddings
            clamped_word_emb, token_ids = clamp_to_nearest_word(pred_word_emb, vocab_embeddings)
            final_x0 = clamped_word_emb + pos_embeddings
        else:
            # We are early in diffusion, let the vectors stay continuous and "blurry"
            final_x0 = pred_x0_combined 
            
        # --- TRUE DDPM POSTERIOR MATH ---
        if t_step > 1:
            t_minus_1 = torch.full((batch_size,), t_step - 1, device=device, dtype=torch.long)
            
            alpha_bar_t = compute_sqrt_alpha_bar(t_tensor).view(-1, 1, 1)
            alpha_bar_t_minus_1 = compute_sqrt_alpha_bar(t_minus_1).view(-1, 1, 1)
            alpha_t = alpha_bar_t / alpha_bar_t_minus_1
            beta_t = 1.0 - alpha_t
            
            c1 = (torch.sqrt(alpha_bar_t_minus_1) * beta_t) / (1.0 - alpha_bar_t)
            c2 = (torch.sqrt(alpha_t) * (1.0 - alpha_bar_t_minus_1)) / (1.0 - alpha_bar_t)
            
            # Use final_x0 (which is either clamped or blurry depending on t_step)
            posterior_mean = c1 * final_x0 + c2 * x_t_emb
            posterior_variance = ((1.0 - alpha_bar_t_minus_1) / (1.0 - alpha_bar_t)) * beta_t
            
            noise = torch.randn_like(x_t_emb)
            x_t_emb = posterior_mean + torch.sqrt(posterior_variance) * noise
        else:
            x_t_emb = final_x0
            
        if t_step % 200 == 0:
            print(f"Denoising step {t_step}/{T_DIFFUSION} complete...")

    # Decode the final token IDs
    if token_ids is not None:
        final_text = tokenizer.batch_decode(token_ids, skip_special_tokens=True)
        print("\n--- Generated Story ---")
        print(final_text[0])
        return final_text
    else:
        print("Error: Clamping never triggered.")

if __name__ == "__main__":
    generate_text("trm_diffusion_epoch_9.pt")

Starting generation reverse loop...
Denoising step 2000/2000 complete...
Denoising step 1800/2000 complete...
Denoising step 1600/2000 complete...
Denoising step 1400/2000 complete...
Denoising step 1200/2000 complete...
Denoising step 1000/2000 complete...
Denoising step 800/2000 complete...
Denoising step 600/2000 complete...
Denoising step 400/2000 complete...
Denoising step 200/2000 complete...

--- Generated Story ---
 and™ but so had have the was't!" loved for go she  in time house park sad too a cat heard. They little made fun., want likeAfter asked was felt One with buy take named
 park to hugged theThe, was took a,
The came. the her
 bigTim get He


In [9]:
import torch
import torch.nn.functional as F
from transformers import GPT2Tokenizer
import math

# 1. Ensure BidirectionalTransformerLayer and TRM_Diffusion are defined here!

def compute_sqrt_alpha_bar(t, T_max=2000, s=1e-4):
    t_norm = t.float() / T_max
    alpha_bar = 1.0 - torch.sqrt(t_norm + s)
    return torch.clamp(alpha_bar, min=1e-5, max=0.9999)

def clamp_to_nearest_word(pred_word_emb, vocab_embeddings):
    dists = torch.cdist(pred_word_emb, vocab_embeddings, p=2) 
    nearest_idx = dists.argmin(dim=-1) 
    clamped_word_emb = vocab_embeddings[nearest_idx]
    return clamped_word_emb, nearest_idx

@torch.no_grad()
def generate_text(model_path, seq_len=64, batch_size=1, temperature=0.8):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    T_DIFFUSION = 2000
    
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    vocab_size = len(tokenizer)
    
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    vocab_embeddings = model.token_emb.weight 
    positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(batch_size, seq_len)
    pos_embeddings = model.pos_emb(positions)
    
    print("Starting generation reverse loop...")
    
    x_t_emb = torch.randn(batch_size, seq_len, model.d_model, device=device)
    
    for t_step in reversed(range(1, T_DIFFUSION + 1)):
        t_tensor = torch.full((batch_size,), t_step, device=device, dtype=torch.long)
        
        z = model.z_init.expand(batch_size, seq_len, -1)
        pred_x0_combined, _, _ = model.deep_recursion(x_t_emb, z, t_tensor, n=6, T=3)
        
        # --- CONSTANT CLAMPING ---
        # Isolate word, clamp it, and re-add positional coords
        pred_word_emb = pred_x0_combined - pos_embeddings
        clamped_word_emb, token_ids = clamp_to_nearest_word(pred_word_emb, vocab_embeddings)
        final_x0 = clamped_word_emb + pos_embeddings
            
        # --- TRUE DDPM POSTERIOR MATH ---
        if t_step > 1:
            t_minus_1 = torch.full((batch_size,), t_step - 1, device=device, dtype=torch.long)
            
            alpha_bar_t = compute_sqrt_alpha_bar(t_tensor).view(-1, 1, 1)
            alpha_bar_t_minus_1 = compute_sqrt_alpha_bar(t_minus_1).view(-1, 1, 1)
            alpha_t = alpha_bar_t / alpha_bar_t_minus_1
            beta_t = 1.0 - alpha_t
            
            c1 = (torch.sqrt(alpha_bar_t_minus_1) * beta_t) / (1.0 - alpha_bar_t)
            c2 = (torch.sqrt(alpha_t) * (1.0 - alpha_bar_t_minus_1)) / (1.0 - alpha_bar_t)
            
            posterior_mean = c1 * final_x0 + c2 * x_t_emb
            posterior_variance = ((1.0 - alpha_bar_t_minus_1) / (1.0 - alpha_bar_t)) * beta_t
            
            # --- TEMPERATURE SCALING ---
            # Shrink the noise slightly so the model doesn't get overwhelmed
            noise = torch.randn_like(x_t_emb) * temperature
            x_t_emb = posterior_mean + torch.sqrt(posterior_variance) * noise
        else:
            x_t_emb = final_x0
            
        if t_step % 200 == 0:
            print(f"Denoising step {t_step}/{T_DIFFUSION} complete...")

    final_text = tokenizer.batch_decode(token_ids, skip_special_tokens=True)
    print("\n--- Generated Story ---")
    print(final_text[0])
    return final_text

if __name__ == "__main__":
    generate_text("trm_diffusion_epoch_9.pt")

Starting generation reverse loop...
Denoising step 2000/2000 complete...
Denoising step 1800/2000 complete...
Denoising step 1600/2000 complete...
Denoising step 1400/2000 complete...
Denoising step 1200/2000 complete...
Denoising step 1000/2000 complete...
Denoising step 800/2000 complete...
Denoising step 600/2000 complete...
Denoising step 400/2000 complete...
Denoising step 200/2000 complete...

--- Generated Story ---
Onceily there
, day, heard heard inside, outside found insideSuddenly he scared in wanted. is scared, she a outside. scared was outside was found.. outside to
 heard decided a inside scared the heard " fun was found little help found,
 it a park. inside on scared " heard outside inside


In [11]:
@torch.no_grad()
def generate_with_prompt(model_path, prompt_text, seq_len=64, temperature=0.8):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    T_DIFFUSION = 2000
    CLAMP_START = 500 
    
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    vocab_size = len(tokenizer)
    
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device)['model_state_dict'])
    model.eval()
    
    vocab_embeddings = model.token_emb.weight 
    positions = torch.arange(seq_len, device=device).unsqueeze(0)
    pos_embeddings = model.pos_emb(positions)
    
    # ==========================================
    # 1. PREPARE THE PROMPT
    # ==========================================
    prompt_tokens = tokenizer(prompt_text, return_tensors="pt")["input_ids"].to(device)
    prompt_len = prompt_tokens.shape[1]
    
    # Get the clean, exact embeddings for the prompt (Word + Position)
    clean_prompt_emb = model.token_emb(prompt_tokens) + pos_embeddings[:, :prompt_len, :]
    
    print(f"Prompt length: {prompt_len} tokens. Generating remaining {seq_len - prompt_len} tokens...")
    
    # Start with pure Gaussian Noise for the whole sequence
    x_t_emb = torch.randn(1, seq_len, model.d_model, device=device)
    
    for t_step in reversed(range(1, T_DIFFUSION + 1)):
        t_tensor = torch.full((1,), t_step, device=device, dtype=torch.long)
        
        # ==========================================
        # 2. FORCE THE PROMPT INTO THE NOISE
        # ==========================================
        # We calculate exactly how much noise the prompt *should* have at this timestep
        alpha_bar_t = compute_sqrt_alpha_bar(t_tensor).view(-1, 1, 1)
        prompt_noise = torch.randn_like(clean_prompt_emb)
        
        # Create the mathematically accurate noised prompt
        noised_prompt = torch.sqrt(alpha_bar_t) * clean_prompt_emb + torch.sqrt(1 - alpha_bar_t) * prompt_noise
        
        # Overwrite the beginning of the sequence with our noised prompt!
        x_t_emb[:, :prompt_len, :] = noised_prompt
        
        # ==========================================
        # 3. STANDARD TRM DENOISING (Predicts the whole sequence)
        # ==========================================
        z = model.z_init.expand(1, seq_len, -1)
        pred_x0_combined, _, _ = model.deep_recursion(x_t_emb, z, t_tensor, n=6, T=3)
        
        # --- DELAYED CLAMPING ---
        if t_step <= CLAMP_START:
            pred_word_emb = pred_x0_combined - pos_embeddings
            clamped_word_emb, token_ids = clamp_to_nearest_word(pred_word_emb, vocab_embeddings)
            final_x0 = clamped_word_emb + pos_embeddings
            
            # FORCE the prompt tokens to be exactly correct during clamping
            final_x0[:, :prompt_len, :] = clean_prompt_emb
        else:
            final_x0 = pred_x0_combined 
            final_x0[:, :prompt_len, :] = clean_prompt_emb # Keep prompt anchored
            
        # --- DDPM POSTERIOR MATH (Same as before) ---
        if t_step > 1:
            t_minus_1 = torch.full((1,), t_step - 1, device=device, dtype=torch.long)
            alpha_bar_t_minus_1 = compute_sqrt_alpha_bar(t_minus_1).view(-1, 1, 1)
            alpha_t = alpha_bar_t / alpha_bar_t_minus_1
            beta_t = 1.0 - alpha_t
            
            c1 = (torch.sqrt(alpha_bar_t_minus_1) * beta_t) / (1.0 - alpha_bar_t)
            c2 = (torch.sqrt(alpha_t) * (1.0 - alpha_bar_t_minus_1)) / (1.0 - alpha_bar_t)
            
            posterior_mean = c1 * final_x0 + c2 * x_t_emb
            posterior_variance = ((1.0 - alpha_bar_t_minus_1) / (1.0 - alpha_bar_t)) * beta_t
            
            noise = torch.randn_like(x_t_emb) * temperature
            x_t_emb = posterior_mean + torch.sqrt(posterior_variance) * noise
        else:
            x_t_emb = final_x0

    # Decode the final token IDs
    final_text = tokenizer.batch_decode(token_ids, skip_special_tokens=True)
    print("\n--- Generated Story ---")
    print(final_text[0])
    return final_text

# Example Usage:
generate_with_prompt("trm_diffusion_epoch_9.pt", "Once upon a time, there was a little dog named")

Prompt length: 11 tokens. Generating remaining 53 tokens...

--- Generated Story ---
Once upon a time, there was a little dog named lived every he red red inside, three
 upon animals sad to down of see knewWhat started and off
 go the out parks by thingsily things held started biggers quickly scared. a shinyThat can sad knew� away heard andily.WhenBut thought


['Once upon a time, there was a little dog named lived every he red red inside, three\n upon animals sad to down of see knewWhat started and off\n go the out parks by thingsily things held started biggers quickly scared. a shinyThat can sad knew� away heard andily.WhenBut thought']

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
vocab_size = len(tokenizer)
model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
model.load_state_dict(torch.load("C:/Users/MSI-1/Desktop/adnan_final/small diff-succes-modls/trm_diffusion_epoch_9.pt", map_location=device)['model_state_dict'])
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters in the model: {params/1e6:.2f}M")

Total trainable parameters in the model: 39.79M


train longer

In [3]:
import sys
!{sys.executable} -m pip install tqdm

You should consider upgrading via the 'c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [4]:
# ==========================================
# 3. Dataset & Dataloader 
# ==========================================
def get_rocstories_dataloader(batch_size=64, seq_len=64):
    """Loads ROCStories and tokenizes them"""
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    
    # We use a subset/variant of ROCStories available on HuggingFace
    dataset = load_dataset("roneneldan/TinyStories", split="train[:1000000]") 
    
    def tokenize(batch):
        # TinyStories just uses the "text" column
        return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=seq_len, return_tensors="pt")
    
    dataset = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
    dataset.set_format(type="torch", columns=["input_ids"])
    return DataLoader(dataset, batch_size=batch_size, shuffle=True), len(tokenizer)

In [5]:
import os
os.environ["WANDB_START_METHOD"] = "thread"
os.environ["WANDB_MODE"] = "offline"

import torch
import torch.nn.functional as F
import wandb
from tqdm import tqdm # Import tqdm here!

# ==========================================
# 4. Training Loop 
# ==========================================
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seq_len = 128
    batch_size = 16
    T_DIFFUSION = 2000
    EPOCHS = 10 # 1 Million stories = ~937,500 total steps over 15 epochs
    
    N_RECURSIONS = 6
    T_CYCLES = 3 
    
    dataloader, vocab_size = get_rocstories_dataloader(batch_size, seq_len)
    
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    ema = EMA(model, decay=0.999)
    
    wandb.init(
        project="trm-continuous-diffusion",
        name="1M-stories-run",
        mode="offline",                                  # <--- FORCES OFFLINE
        settings=wandb.Settings(start_method="thread"),
        config={
            "dataset_size": "1M",
            "epochs": EPOCHS,
            "batch_size": batch_size,
            "seq_len": seq_len,
            "learning_rate": 1e-4
        }
    )
    
    print("Starting Training...")
    model.train()
    global_step = 0 
    
    for epoch in range(EPOCHS): 
        # Wrap the dataloader in tqdm to create the progress bar
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
        
        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            bs = input_ids.shape[0]
            
            # 1. Clean x_0 Embeddings
            y_true_emb = model.token_emb(input_ids)
            positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(bs, seq_len)
            y_true_emb = y_true_emb + model.pos_emb(positions)
            
            # 2. Sample random timestep t and apply Sqrt Noise
            t = torch.randint(1, T_DIFFUSION + 1, (bs,), device=device)
            alpha_bar_t = compute_sqrt_alpha_bar(t, T_max=T_DIFFUSION).view(bs, 1, 1)
            noise = torch.randn_like(y_true_emb)
            
            # Forward Process q(x_t | x_0)
            x_t_emb = torch.sqrt(alpha_bar_t) * y_true_emb + torch.sqrt(1 - alpha_bar_t) * noise
            
            # 3. TRM Denoiser Prediction
            z = model.z_init.expand(bs, seq_len, -1)
            pred_x0_emb, z, _ = model.deep_recursion(x_t_emb, z, t, n=N_RECURSIONS, T=T_CYCLES)
            
            # 4. Calculate Diffusion-LM Losses
            loss_recon = F.mse_loss(pred_x0_emb, y_true_emb)
            loss_emb_anchor = F.mse_loss(y_true_emb, pred_x0_emb.detach())
            total_loss = loss_recon + (0.1 * loss_emb_anchor)
            
            # 5. Backpropagate
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ema.update()
            
            # ==========================================
            # UPDATE METRICS LIVE
            # ==========================================
            wandb.log({
                "train/total_loss": total_loss.item(),
                "train/loss_recon": loss_recon.item(),
                "epoch": epoch,
                "global_step": global_step
            })
            
            # Updates the right side of the progress bar with the current loss
            if step % 10 == 0:
                pbar.set_postfix({'Loss': f"{total_loss.item():.4f}"})
                
            # Mid-epoch save
            if global_step > 0 and global_step % 10000 == 0:
                ema.apply_shadow()
                save_path = f"trm_diffusion_step_{global_step}.pt"
                torch.save({
                    'epoch': epoch,
                    'global_step': global_step,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                }, save_path)
                wandb.save(save_path) 
                ema.restore()
                
            global_step += 1

        print(f"--- Epoch {epoch+1} Complete ---")
        ema.apply_shadow()
        save_path = f"trm_diffusion_epoch_{epoch+1}_complete.pt"
        torch.save({
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, save_path)
        wandb.save(save_path) 
        ema.restore()
        
    wandb.finish()

if __name__ == "__main__":
    train()

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.


ServicePollForTokenError: Failed to read port info after 30.0 seconds.

In [10]:
import sys
!{sys.executable} -m pip install tensorboard

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
You should consider upgrading via the 'c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [11]:
import torch
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter # <-- Built-in PyTorch logger

def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seq_len = 128
    batch_size = 16
    T_DIFFUSION = 2000
    EPOCHS = 10 
    
    N_RECURSIONS = 6
    T_CYCLES = 3 
    
    dataloader, vocab_size = get_rocstories_dataloader(batch_size, seq_len)
    
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    ema = EMA(model, decay=0.999)
    
    # Initialize TensorBoard Writer
    writer = SummaryWriter(log_dir="runs/1M_stories_diffusion")
    
    print("Starting Training with TensorBoard...")
    model.train()
    global_step = 0 
    
    for epoch in range(EPOCHS): 
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
        
        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            bs = input_ids.shape[0]
            
            y_true_emb = model.token_emb(input_ids)
            positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(bs, seq_len)
            y_true_emb = y_true_emb + model.pos_emb(positions)
            
            t = torch.randint(1, T_DIFFUSION + 1, (bs,), device=device)
            alpha_bar_t = compute_sqrt_alpha_bar(t, T_max=T_DIFFUSION).view(bs, 1, 1)
            noise = torch.randn_like(y_true_emb)
            
            x_t_emb = torch.sqrt(alpha_bar_t) * y_true_emb + torch.sqrt(1 - alpha_bar_t) * noise
            
            z = model.z_init.expand(bs, seq_len, -1)
            pred_x0_emb, z, _ = model.deep_recursion(x_t_emb, z, t, n=N_RECURSIONS, T=T_CYCLES)
            
            loss_recon = F.mse_loss(pred_x0_emb, y_true_emb)
            loss_emb_anchor = F.mse_loss(y_true_emb, pred_x0_emb.detach())
            total_loss = loss_recon + (0.1 * loss_emb_anchor)
            
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ema.update()
            
            # Log to TensorBoard
            writer.add_scalar("Loss/Total", total_loss.item(), global_step)
            writer.add_scalar("Loss/Reconstruction", loss_recon.item(), global_step)
            
            if step % 10 == 0:
                pbar.set_postfix({'Loss': f"{total_loss.item():.4f}"})
                
            if global_step > 0 and global_step % 10000 == 0:
                ema.apply_shadow()
                save_path = f"trm_diffusion_step_{global_step}.pt"
                torch.save({
                    'epoch': epoch,
                    'global_step': global_step,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                }, save_path)
                ema.restore()
                
            global_step += 1

        print(f"--- Epoch {epoch+1} Complete ---")
        ema.apply_shadow()
        save_path = f"trm_diffusion_epoch_{epoch+1}_complete.pt"
        torch.save({
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, save_path)
        ema.restore()
        
    writer.close()

if __name__ == "__main__":
    train()

Starting Training with TensorBoard...


Epoch 1/10: 100%|██████████| 62500/62500 [13:31:20<00:00,  1.28it/s, Loss=0.0144]  


--- Epoch 1 Complete ---


Epoch 2/10:   1%|          | 545/62500 [06:24<12:07:38,  1.42it/s, Loss=0.0163]


KeyboardInterrupt: 

In [ ]:
!tensorboard --logdir=runs

In [10]:
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer

def get_rocstories_dataloader(batch_size, seq_len):
    # Load dataset from HuggingFace
    dataset = load_dataset("mintujupally/ROCStories", split='train')
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    def tokenize_function(examples):
        # Concatenate sentences 1-5 into a single story
        
        return tokenizer(
            examples['text'], 
            padding="max_length", 
            truncation=True, 
            max_length=seq_len, 
            return_tensors="pt"
        )

    tokenized_datasets = dataset.map(
        tokenize_function, 
        batched=True, 
        remove_columns=dataset.column_names
    )
    tokenized_datasets.set_format("torch")

    dataloader = DataLoader(tokenized_datasets, batch_size=batch_size, shuffle=True, num_workers=4)
    return dataloader, len(tokenizer)

In [11]:
dataloader, vocab_size = get_rocstories_dataloader(128, 64)

Repo card metadata block was not found. Setting CardData to empty.
Map: 100%|██████████| 78528/78528 [00:04<00:00, 17805.64 examples/s]


In [12]:
next(iter(dataloader))

{'input_ids': tensor([[43187,  2227,   284,  ..., 50256, 50256, 50256],
         [23865,   373,  5059,  ..., 50256, 50256, 50256],
         [30746,   494,   373,  ..., 50256, 50256, 50256],
         ...,
         [   43,   813,   373,  ..., 50256, 50256, 50256],
         [13787,   373,  2111,  ..., 50256, 50256, 50256],
         [   40,   373,  4964,  ..., 50256, 50256, 50256]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]])}

In [16]:
import torch
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter # <-- Built-in PyTorch logger

def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seq_len = 128
    batch_size = 16
    T_DIFFUSION = 2000
    EPOCHS = 10 
    
    N_RECURSIONS = 6
    T_CYCLES = 3 
    
    dataloader, vocab_size = get_rocstories_dataloader(batch_size, seq_len)
    
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    ema = EMA(model, decay=0.999)
    
    # Initialize TensorBoard Writer
    writer = SummaryWriter(log_dir="runs/1M_stories_diffusion")
    
    print("Starting Training with TensorBoard...")
    model.train()
    global_step = 0 
    
    for epoch in range(EPOCHS): 
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
        
        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            bs = input_ids.shape[0]
            
            y_true_emb = model.token_emb(input_ids)
            positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(bs, seq_len)
            y_true_emb = y_true_emb + model.pos_emb(positions)
            
            t = torch.randint(1, T_DIFFUSION + 1, (bs,), device=device)
            alpha_bar_t = compute_sqrt_alpha_bar(t, T_max=T_DIFFUSION).view(bs, 1, 1)
            noise = torch.randn_like(y_true_emb)
            
            x_t_emb = torch.sqrt(alpha_bar_t) * y_true_emb + torch.sqrt(1 - alpha_bar_t) * noise
            
            z = model.z_init.expand(bs, seq_len, -1)
            pred_x0_emb, z, _ = model.deep_recursion(x_t_emb, z, t, n=N_RECURSIONS, T=T_CYCLES)
            
            loss_recon = F.mse_loss(pred_x0_emb, y_true_emb)
            loss_emb_anchor = F.mse_loss(y_true_emb, pred_x0_emb.detach())
            total_loss = loss_recon + (0.1 * loss_emb_anchor)
            
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ema.update()
            
            # Log to TensorBoard
            writer.add_scalar("Loss/Total", total_loss.item(), global_step)
            writer.add_scalar("Loss/Reconstruction", loss_recon.item(), global_step)
            
            if step % 10 == 0:
                pbar.set_postfix({'Loss': f"{total_loss.item():.4f}"})
                
            if global_step > 0 and global_step % 10000 == 0:
                ema.apply_shadow()
                save_path = f"trm_diffusion_step_{global_step}.pt"
                torch.save({
                    'epoch': epoch,
                    'global_step': global_step,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                }, save_path)
                ema.restore()
                
            global_step += 1

        print(f"--- Epoch {epoch+1} Complete ---")
        ema.apply_shadow()
        save_path = f"trm_diffusion_epoch_{epoch+1}_complete.pt"
        torch.save({
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, save_path)
        ema.restore()
        
    writer.close()

if __name__ == "__main__":
    train()

Repo card metadata block was not found. Setting CardData to empty.


Starting Training with TensorBoard...


Epoch 1/10:   0%|          | 19/4908 [00:18<1:17:45,  1.05it/s, Loss=1.8100]


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from transformers import GPT2Tokenizer
from datasets import load_dataset
import wandb
from tqdm import tqdm
import os

def train_roc_diffusion():
    # --- Hyperparameters & Config ---
    device = torch.device("cuda")
    seq_len = 64
    batch_size = 128  # Optimized for L40S 48GB VRAM
    epochs = 10
    lr = 1e-4
    save_dir = "/workspace/checkpoints" # Standard RunPod persistent path
    os.makedirs(save_dir, exist_ok=True)

    # Initialize Dataset & Model
    dataloader, vocab_size = get_rocstories_dataloader(batch_size, seq_len)
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=6).to(device)
    
    # Optimizer & Scaler
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scaler = GradScaler() 
    ema = EMA(model, decay=0.9999) # Diffusion-LM requires EMA for generation quality

    # --- WandB Initialization ---
    wandb.init(
        project="diffusion-lm-roc",
        name="L40S-ROCStories-Run1",
        config={
            "batch_size": batch_size,
            "learning_rate": lr,
            "epochs": epochs,
            "architecture": "TRM-Diffusion",
            "gpu": "NVIDIA L40S"
        }
    )

    global_step = 0
    print(f"Starting Training on {device}...")

    for epoch in range(epochs):
        model.train()
        pbar = tqdm(dataloader, desc=f"Epoch {epoch}")
        
        for batch in pbar:
            input_ids = batch["input_ids"].to(device)
            
            # 1. Get Target Embeddings
            with torch.no_grad():
                y_true_emb = model.token_emb(input_ids)
                positions = torch.arange(seq_len, device=device).unsqueeze(0)
                y_true_emb = y_true_emb + model.pos_emb(positions)

            # 2. Add Diffusion Noise
            t = torch.randint(1, 2001, (input_ids.shape[0],), device=device)
            alpha_bar_t = compute_sqrt_alpha_bar(t).view(-1, 1, 1)
            noise = torch.randn_like(y_true_emb)
            x_t_emb = torch.sqrt(alpha_bar_t) * y_true_emb + torch.sqrt(1 - alpha_bar_t) * noise

            # 3. Forward Pass (Mixed Precision)
            optimizer.zero_grad()
            with autocast(dtype=torch.bfloat16):
                z = model.z_init.expand(input_ids.shape[0], seq_len, -1)
                pred_x0_emb, _, _ = model.deep_recursion(x_t_emb, z, t, n=6, T=3)
                
                # Combined Loss (MSE + Embedding Anchor)
                loss_recon = F.mse_loss(pred_x0_emb, y_true_emb)
                loss_anchor = F.mse_loss(y_true_emb, pred_x0_emb.detach())
                total_loss = loss_recon + (0.1 * loss_anchor)

            # 4. Scaled Backward Pass
            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            
            # Update EMA weights
            ema.update()

            # Logging
            if global_step % 20 == 0:
                wandb.log({
                    "train/loss": total_loss.item(),
                    "train/recon_loss": loss_recon.item(),
                    "train/lr": optimizer.param_groups[0]['lr'],
                    "global_step": global_step
                })
                pbar.set_postfix({"Loss": f"{total_loss.item():.4f}"})
            
            global_step += 1

        # --- END OF EPOCH SAVING ---
        print(f"Epoch {epoch} finished. Saving EMA checkpoint...")
        
        # We save the EMA weights because they are much better for sampling
        ema.apply_shadow()
        
        save_path = os.path.join(save_dir, f"diffusion_roc_epoch_{epoch}.pt")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'global_step': global_step,
            'vocab_size': vocab_size
        }, save_path)
        
        # Important: Restore original weights so training continues naturally
        ema.restore()
        
        print(f"Saved: {save_path}")

    wandb.finish()
    print("Training Complete!")

if __name__ == "__main__":
    train_roc_diffusion()